# Processamento da Camada Gold

## 1. Objetivo

Construir tabelas analíticas a partir dos dados tratados e validados na camada Silver, disponibilizando informações consolidadas para consultas e análises de negócio.

## 2. Estratégia de processamento

A transformação é implementada no módulo `src/gold.py`. Este notebook atua como camada de orquestração e validação, mantendo a lógica de negócio reutilizável fora do notebook.

Serão construídas duas tabelas Delta:

- `gold_transacoes_pix`: visão detalhada das transações PIX, enriquecida com informações de contas e clientes.
- `gold_resumo_cliente`: indicadores financeiros consolidados por cliente, incluindo clientes sem transações.

As tabelas de origem estão no schema `workspace.silver` e os resultados serão gravados em `workspace.gold`.

## 3. Execução da pipeline Gold

A função `run_gold()` realiza a leitura das tabelas Silver, constrói as duas visões analíticas e persiste os resultados no formato Delta.

In [0]:
from importlib import reload
import src.gold

reload(src.gold)

resultados = src.gold.run_gold(spark)

## 4. Validação das tabelas Gold

Após a execução, as tabelas são consultadas novamente no catálogo para validar os dados efetivamente persistidos.

In [0]:
from pyspark.sql import functions as F

GOLD_SCHEMA = "workspace.gold"

df_gold_transacoes = spark.table(
    f"{GOLD_SCHEMA}.gold_transacoes_pix"
)

df_gold_resumo = spark.table(
    f"{GOLD_SCHEMA}.gold_resumo_cliente"
)

print("Transações PIX:", df_gold_transacoes.count())
print("Clientes:", df_gold_resumo.count())

print("\nSchema gold_transacoes_pix:")
df_gold_transacoes.printSchema()

print("\nSchema gold_resumo_cliente:")
df_gold_resumo.printSchema()

## 5. Verificações de qualidade

As verificações abaixo conferem os volumes esperados, a unicidade das chaves e a preservação dos clientes sem transações.

In [0]:
total_transacoes = df_gold_transacoes.count()
total_clientes = df_gold_resumo.count()

transacoes_duplicadas = (
    df_gold_transacoes
    .groupBy("id_transacao")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

clientes_duplicados = (
    df_gold_resumo
    .groupBy("id_cliente")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

clientes_sem_transacoes = (
    df_gold_resumo
    .filter(F.col("quantidade_transacoes") == 0)
    .count()
)

print("Transações duplicadas:", transacoes_duplicadas)
print("Clientes duplicados:", clientes_duplicados)
print("Clientes sem transações:", clientes_sem_transacoes)

assert total_transacoes == 147, "Volume inesperado em gold_transacoes_pix"
assert total_clientes == 24, "Volume inesperado em gold_resumo_cliente"
assert transacoes_duplicadas == 0, "Existem transações duplicadas na Gold"
assert clientes_duplicados == 0, "Existem clientes duplicados na Gold"

print("\nValidações Gold concluídas com sucesso.")

## 6. Inspeção dos resultados

### Transações PIX enriquecidas

Exibe uma amostra da visão detalhada das transações.

In [0]:
display(df_gold_transacoes.orderBy("id_transacao").limit(10))

### Resumo financeiro por cliente

Exibe os clientes ordenados pelo maior volume financeiro movimentado.

In [0]:
display(
    df_gold_resumo
    .orderBy(F.desc("total_transacionado"))
)


## 7. Resultado

A camada Gold disponibiliza duas tabelas Delta para consumo analítico:

- `workspace.gold.gold_transacoes_pix`
- `workspace.gold.gold_resumo_cliente`

A execução é centralizada em `src/gold.py`, enquanto este notebook documenta a orquestração, a persistência e as validações dos resultados.